In [8]:
!pip install kagglehub



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import kagglehub

# Download latest version
path_ = kagglehub.dataset_download("gauravmalik26/food-delivery-dataset")





In [10]:
import os

print(os.listdir(path_))

['test.csv', 'train.csv', 'Sample_Submission.csv']


In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

df_train = pd.read_csv(os.path.join(path_, "train.csv"))
df_test= pd.read_csv(os.path.join(path_, "test.csv"))



# X_train = df_train[[""]]

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from numpy import radians, sin, cos, sqrt, arctan2
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * arctan2(sqrt(a), sqrt(1 - a))

for df in [df_train, df_test]:
    df["distance_km"] = haversine(
        df["Restaurant_latitude"], df["Restaurant_longitude"],
        df["Delivery_location_latitude"], df["Delivery_location_longitude"]
    )

drop_cols = ["Restaurant_latitude", "Restaurant_longitude",
             "Delivery_location_latitude", "Delivery_location_longitude"]

df_train = df_train.drop(columns=drop_cols)
df_test  = df_test.drop(columns=drop_cols)

X_train = df_train.drop(columns=["Time_taken(min)"])
y_train = df_train["Time_taken(min)"]
X_test  = df_test.drop(columns=["Time_taken(min)"])
y_test  = df_test["Time_taken(min)"]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
cv_mae  = mae  / y_test.std()
cv_rmse = rmse / y_test.std()

print(f"MAE:      {mae:.4f}")
print(f"RMSE:     {rmse:.4f}")
print(f"MAPE:     {mape:.2f}%")
print(f"MAE/std:  {cv_mae:.4f}")
print(f"RMSE/std: {cv_rmse:.4f}")


KeyError: "['Time_taken(min)'] not found in axis"